# Notebook 03: Bayesian Inference Demo

**Gap 2**: Given an observed polarization spectrum, recover the posterior
distribution over (spin, r_bp, β, φ, inclination).


In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from src.inference.run_mcmc import log_posterior, summarize_posterior
from src.emulator.model import load_emulator
from src.utils.physics import PARAM_BOUNDS, ENERGY_BINS
from src.utils.io import load_dataset_hdf5


In [ ]:
# Create a synthetic 'observed' spectrum from known parameters
# (Ground truth test: check if MCMC recovers the correct values)
from src.parameter_sweep.generate_grid import mock_simulate

true_params = {'spin': 0.9, 'r_bp': 8.0, 'beta': 15.0, 'phi': 90.0, 'inclination': 75.0}
true_spectrum = mock_simulate(**true_params)  # (N_energy, 3)

# Add observational noise (SNR=20)
noisy = true_spectrum.copy()
noisy[:, 1] += np.random.normal(0, 0.5, len(ENERGY_BINS))   # pol_frac noise
noisy[:, 2] += np.random.normal(0, 2.0, len(ENERGY_BINS))   # pol_angle noise
observed = noisy[:, 1:]  # (N_energy, 2)

print('True parameters:', true_params)


In [ ]:
# Run MCMC (requires trained emulator from Notebook 02)
import emcee
from src.inference.run_mcmc import run_emcee

emulator, normalizer = load_emulator('../results/models/emulator_best.pt')
samples, sampler = run_emcee(observed, emulator, normalizer, n_walkers=16, n_steps=500)
print(f'Posterior samples shape: {samples.shape}')


In [ ]:
# Summarize and plot
param_names = list(PARAM_BOUNDS.keys())
print('\nPosterior summary (compare to true values above):')
results = summarize_posterior(samples, param_names)

try:
    import corner
    truths = list(true_params.values())
    fig = corner.corner(samples, labels=param_names, truths=truths,
                        truth_color='red', show_titles=True)
    plt.savefig('../results/figures/corner_plot.png', dpi=120)
    plt.show()
except ImportError:
    print('pip install corner  for the posterior corner plot')
